# 00 - Generate Sample Events

Creates simulated customer activity events and writes them as JSON to the raw landing path. This simulates source events produced by Kafka.

In [ ]:
%run ../utilities/config

In [ ]:
from pyspark.sql import Row
from datetime import datetime, timezone
import random
import uuid

NUMBER_OF_EVENTS = 1000

events = []

for _ in range(NUMBER_OF_EVENTS):
    event_type = random.choice(VALID_EVENT_TYPES)

    event = {
        "event_id": str(uuid.uuid4()),
        "user_id": f"U{random.randint(1000, 9999)}",
        "session_id": str(uuid.uuid4()),
        "event_type": event_type,
        "product_id": None,
        "category": None,
        "price": None,
        "quantity": None,
        "payment_method": None,
        "event_time": datetime.now(timezone.utc).isoformat(),
        "source": "web_app",
    }

    if event_type in ["product_view", "cart_add", "checkout", "payment_success", "payment_failed"]:
        event["product_id"] = f"P{random.randint(100, 999)}"
        event["category"] = random.choice(["electronics", "fashion", "beauty", "home", "grocery", "books"])
        event["price"] = round(random.uniform(10, 500), 2)
        event["quantity"] = random.randint(1, 5)

    if event_type in ["payment_success", "payment_failed"]:
        event["payment_method"] = random.choice(["credit_card", "debit_card", "paypal", "gift_card"])

    events.append(Row(**event))

raw_df = spark.createDataFrame(events)

raw_df.write.mode("overwrite").json(RAW_PATH)

display(spark.read.json(RAW_PATH))